# Лабораторная работа №3
## Вариант 3.2
#### Выполнили: Аверьянова Мария, Калягин Дмитрий, Кашникова Анна, Климович Анна

## Исходная задача

Рассмотрим задачу минимизации:

$$\begin{aligned}
\text{minimize} \quad & f(x) = \sum_{i=1}^n x_i \log x_i \\
\text{subject to} \quad & Ax = b \\
& x \geq \mathbf{1}
\end{aligned}$$

где $A \in \mathbb{R}^{p \times n}$, $b \in \mathbb{R}^p$, $p < n$.

---






#### **ЗАДАНИЕ 1: Логарифмический барьер и исследование на выпуклость**

**1.1. Прямая задача с логарифмическим барьером**

Неравенства: $g_i(x) = 1 - x_i \leq 0$ (так как $x_i \geq 1$)

Логарифмический барьер:
$$B_L(x) = -\sum_{i=1}^n \log(x_i - 1)$$

Барьерная задача для фиксированного $t > 0$:
$$\min_x \left\{ t\sum_{i=1}^n x_i \log x_i - \sum_{i=1}^n \log(x_i - 1) \right\}$$
$$\text{subject to } Ax = b$$

**Исследование на выпуклость:**

Функция $f(x) = \sum_{i=1}^n x_i \log x_i$ выпукла, так как:
- $\frac{d^2}{dx_i^2}(x_i \log x_i) = \frac{1}{x_i} > 0$ при $x_i > 0$

Барьерная функция $-\log(x_i - 1)$ выпукла при $x_i > 1$, так как:
- $\frac{d^2}{dx_i^2}[-\log(x_i - 1)] = \frac{1}{(x_i-1)^2} > 0$

**Вывод:** Барьерная задача выпукла (сумма выпуклых функций).


# 2. Генерация тестовых примеров и эталонное решение через CVXPY

In [1]:
import time
import pickle
import matplotlib.pyplot as plt
import numpy as np
import cvxpy as cp
import pandas as pd
from pathlib import Path
from scipy.linalg import svd
from scipy.optimize import minimize
from typing import List, Dict, Tuple
from scipy.sparse.linalg import cg

In [2]:
def entropy_objective(x: np.ndarray) -> float:
    if np.any(x <= 0):
        return np.inf
    return np.sum(x * np.log(x))

def recover_primal_from_dual(nu: np.ndarray, lam: np.ndarray, A: np.ndarray) -> np.ndarray:
    # return np.exp(-A.T @ nu - 1)
    return np.exp(-1 - A.T @ nu + lam)

def generate_full_rank_matrix(p: int, n: int, rng) -> np.ndarray:
    while True:
        A = rng.standard_normal((p, n))
        if np.linalg.matrix_rank(A) == p:
            return A

def nullspace_basis(A: np.ndarray, tol=1e-12) -> np.ndarray:
    U, S, Vt = svd(A, full_matrices=True)
    rank = np.sum(S > tol * S[0])
    return Vt[rank:].T

ГЕНЕРАЦИЯ ЗАДАЧИ

In [3]:
def generate_test_problem(n: int, p: int, seed: int):
    rng = np.random.default_rng(seed)

    A = generate_full_rank_matrix(p, n, rng)

    # Генерация допустимой точки x >= 1
    x_feas = 1.0 + np.abs(rng.standard_normal(n))  # x >= 1

    b = A @ x_feas

    N = nullspace_basis(A)

    return {
        'A': A,
        'b': b,
        'x_feas': x_feas,
        'N': N,
        'rng': rng,
        'n': n,
        'p': p
    }

СТАРТОВЫЕ ТОЧКИ

In [4]:
def generate_feasible_starts(x_ref, N, num_starts, rng):
    starts = []
    n_p = N.shape[1]

    if n_p == 0:
        return [x_ref.copy() for _ in range(num_starts)]

    for _ in range(num_starts):
        z = rng.standard_normal(n_p)
        direction = N @ z

        alpha_max = np.inf
        mask = direction < 0
        if np.any(mask):
            alpha_max = np.min((x_ref[mask] - 1.0) / (-direction[mask])) * 0.9

        alpha = alpha_max * (0.2 + 0.6 * rng.random()) if np.isfinite(alpha_max) else 0.0

        x0 = x_ref + alpha * direction
        x0 = np.maximum(x0, 1.0 + 1e-8)

        starts.append(x0)

    return starts

CVXPY РЕШЕНИЕ

In [5]:
def solve_primal(A, b):

    n = A.shape[1]
    x = cp.Variable(n)

    objective = cp.Minimize(cp.sum(-cp.entr(x)))

    constraints = [A @ x == b, x >= 1.0]

    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.SCS, eps=1e-8, max_iters=10000)

    if x.value is None:
        raise RuntimeError("Primal solve failed")

    x_star = x.value
    f_star = np.sum(x_star * np.log(x_star))

    return x_star, f_star


def solve_dual_cvxpy(A, b):

    p, n = A.shape

    nu = cp.Variable(p)

    lam = cp.Variable(n, nonneg=True)

    phi_expr = (
        b @ nu
        - cp.sum(lam)
        + cp.sum(
            cp.exp(-1 - A.T @ nu + lam)
        )
    )

    prob = cp.Problem(cp.Minimize(phi_expr))

    prob.solve(
        solver=cp.SCS,
        eps=1e-8,
        max_iters=10000
    )

    if nu.value is None or lam.value is None:
        raise RuntimeError("Dual solve failed")

    nu_star = nu.value

    lam_star = lam.value

    phi_star = prob.value

    q_star = -phi_star

    return nu_star, lam_star, q_star, phi_star

In [6]:
def build_dataset(
    n_values=range(10, 101, 10),
    num_instances=5,
    num_starts=5,
    base_seed=1000,
    save_path=None,
):
    problems: List[Dict] = []
    global_problem_id = 0


    for n in n_values:
        p = n // 2  # p < n

        print(f"\n>>> n = {n}, p = {p}")

        for inst_id in range(num_instances):
            seed = base_seed * n + inst_id

            prob = generate_test_problem(n, p, seed)
            A = prob["A"]
            b = prob["b"]
            N = prob["N"]
            rng = prob["rng"]

            # CVXPY reference solutions
            x_star, f_star = solve_primal(A, b)
            nu_star, lam_star, q_star, phi_star = solve_dual_cvxpy(A, b)

            x_dual = recover_primal_from_dual(nu_star, lam_star, A)
            duality_gap = abs(f_star - q_star)

            # primal feasible starts
            primal_starts = generate_feasible_starts(
                prob["x_feas"], N, num_starts, rng
            )

            # dual starts
            dual_starts = [
                rng.standard_normal(p) * 0.1
                for _ in range(num_starts)
            ]


            problems.append({
                "problem_id": global_problem_id,
                "n": n,
                "p": p,
                "instance_id": inst_id,
                "seed": seed,
                "A": A,
                "b": b,
                "N": N,
                "x_feas": prob["x_feas"],
                "x_star": x_star,
                "f_star": f_star,
                "nu_star": nu_star,
                "lam_star" : lam_star,
                "q_star": q_star,
                "phi_star": phi_star,
                "x_dual": x_dual,
                "duality_gap": duality_gap,
                "primal_starts": primal_starts,
                "dual_starts": dual_starts,
            })

            print(f"  Instance {inst_id}: f* = {f_star:.4f}, "
                  f"q* = {q_star:.4f}, gap = {duality_gap:.2e}")

            global_problem_id += 1

    if save_path is not None:
        with open(save_path, "wb") as f:
            pickle.dump(problems, f)
        print(f"\nDataset saved to {save_path}")

    return problems

In [7]:
problems = build_dataset(
    n_values=range(10, 101, 10),
    num_instances=5,
    num_starts=5,
    save_path="entropy_problems.pkl",
)


>>> n = 10, p = 5
  Instance 0: f* = 8.0303, q* = 8.0303, gap = 2.55e-08
  Instance 1: f* = 3.9743, q* = 3.9743, gap = 1.06e-08
  Instance 2: f* = 10.1532, q* = 10.1532, gap = 6.96e-08
  Instance 3: f* = 6.2867, q* = 6.2867, gap = 4.33e-09
  Instance 4: f* = 8.6044, q* = 8.6044, gap = 1.60e-08

>>> n = 20, p = 10
  Instance 0: f* = 16.1201, q* = 16.1201, gap = 1.52e-08
  Instance 1: f* = 17.5840, q* = 17.5840, gap = 4.58e-08
  Instance 2: f* = 23.4347, q* = 23.4347, gap = 1.36e-07
  Instance 3: f* = 10.4536, q* = 10.4536, gap = 3.34e-08
  Instance 4: f* = 14.3121, q* = 14.3121, gap = 2.84e-07

>>> n = 30, p = 15
  Instance 0: f* = 21.3158, q* = 21.3158, gap = 2.66e-08
  Instance 1: f* = 29.8398, q* = 29.8398, gap = 1.17e-09
  Instance 2: f* = 17.1844, q* = 17.1844, gap = 1.92e-07
  Instance 3: f* = 23.0946, q* = 23.0946, gap = 5.22e-08
  Instance 4: f* = 14.1162, q* = 14.1162, gap = 9.99e-09

>>> n = 40, p = 20
  Instance 0: f* = 30.8651, q* = 30.8651, gap = 1.08e-07
  Instance 1: f* 

#ЗАДАНИЕ 3a

In [8]:
def line_search_scalar(
    f,
    x,
    p,
    gamma=1.0,
    min_gamma=1e-8,
    num=30,
):
    f_x = f(x)

    if not np.isfinite(f_x):
        return 0.0

    gammas = np.geomspace(gamma, min_gamma, num)

    best_gamma = 0.0
    best_value = f_x

    for c_gamma in gammas:
        value = f(x + c_gamma * p)

        if np.isfinite(value) and value < best_value:
            best_value = value
            best_gamma = c_gamma

    return best_gamma

In [9]:
def barrier_function_primal(x: np.ndarray, t: float) -> float:
    if np.any(x <= 1):
        return np.inf

    entropy_term = np.sum(x * np.log(x))
    barrier_term = np.sum(np.log(x - 1))

    return t * entropy_term - barrier_term

def barrier_gradient_primal(x: np.ndarray, t: float) -> np.ndarray:
    grad = t * (np.log(x) + 1) - 1.0 / (x - 1)
    return grad

def barrier_hessian_primal(x: np.ndarray, t: float) -> np.ndarray:
    diag = t / x + 1.0 / (x - 1)**2
    return np.diag(diag)

def newton_step_equality_constrained(
    x: np.ndarray, t: float, A: np.ndarray, b: np.ndarray
):

    n = len(x)
    p = A.shape[0]

    grad = barrier_gradient_primal(x, t)
    H = barrier_hessian_primal(x, t)

    KKT = np.block([
        [H, A.T],
        [A, np.zeros((p, p))]
    ])

    rhs = -np.concatenate([grad, A @ x - b])

    try:
        sol = np.linalg.solve(KKT, rhs)
        dx = sol[:n]
        return dx
    except np.linalg.LinAlgError:
        sol = np.linalg.lstsq(KKT, rhs, rcond=None)[0]
        return sol[:n]

def barrier_method_newton(
    problem, x0, mu = 10.0,
    t0 = 1.0, eps = 0.01, max_iter = 100
) :

    A = problem['A']
    b = problem['b']
    f_star = problem['f_star']
    n = problem['n']
    m = n

    x = x0.copy()
    t = t0

    history = {
        'outer_iter': [],
        'inner_iter': [],
        'objective': [],
        'error': [],
        't_values': [],
        'total_newton_steps': 0
    }

    total_newton_steps = 0

    for outer_iter in range(max_iter):
        # Центрирующий шаг: минимизируем tf + phi методом Ньютона
        inner_iter = 0
        newton_decrement = np.inf

        while newton_decrement > 1e-6 and inner_iter < 50:
            dx = newton_step_equality_constrained(x, t, A, b)

            H = barrier_hessian_primal(x, t)
            newton_decrement = np.sqrt(np.abs(dx @ H @ dx))

            if newton_decrement < 1e-6:
                break

            step = line_search_scalar(
                f=lambda x_: barrier_function_primal(x_, t),
                x=x,
                p=dx,
                gamma=1.0,
                min_gamma=1e-8,
                num=30
            )

            if step <= 1e-12:
                print(f"Line search failed at iter {outer_iter + 1}")
                break

            x = x + step * dx
            x = np.maximum(x, 1.0 + 1e-10)

            inner_iter += 1
            total_newton_steps += 1


        f_val = np.sum(x * np.log(x))
        error = abs(f_val - f_star)
        duality_gap = m / t

        history['outer_iter'].append(outer_iter)
        history['inner_iter'].append(inner_iter)
        history['objective'].append(f_val)
        history['error'].append(error)
        history['t_values'].append(t)
        history['total_newton_steps'] = total_newton_steps


        if error <= eps:

            break

        t *= mu

    return x, f_val, history

In [10]:
def dual_phi(
    z: np.ndarray,
    A: np.ndarray,
    b: np.ndarray,
    t: float
) -> float:

    p = A.shape[0]

    nu = z[:p]

    lam = z[p:]

    if np.any(lam <= 0):
        return np.inf

    exp_term = np.exp(
        -1 - A.T @ nu + lam
    )

    objective = (
        t * (
            b @ nu
            - np.sum(lam)
            + np.sum(exp_term)
        )
        - np.sum(np.log(lam))
    )

    return objective


def dual_phi_grad(
    z: np.ndarray,
    A: np.ndarray,
    b: np.ndarray,
    t: float
) -> np.ndarray:

    p = A.shape[0]

    nu = z[:p]

    lam = z[p:]

    exp_term = np.exp(
        -1 - A.T @ nu + lam
    )

    grad_nu = t * (
        b - A @ exp_term
    )

    grad_lam = (
        t * (-1.0 + exp_term)
        - 1.0 / lam
    )

    return np.concatenate([
        grad_nu,
        grad_lam
    ])


def dual_phi_hess(
    z: np.ndarray,
    A: np.ndarray,
    t: float
) -> np.ndarray:

    p, n = A.shape

    nu = z[:p]

    lam = z[p:]

    exp_term = np.exp(
        -1 - A.T @ nu + lam
    )

    D = np.diag(exp_term)

    H_nu_nu = t * (
        A @ D @ A.T
    )

    H_lam_lam = (
        t * D
        + np.diag(1.0 / lam**2)
    )

    H_nu_lam = -t * (
        A @ D
    )

    H_lam_nu = H_nu_lam.T

    H = np.block([
        [H_nu_nu, H_nu_lam],
        [H_lam_nu, H_lam_lam]
    ])

    return H



def barrier_method_dual_newton(
    problem: Dict,
    nu0: np.ndarray,
    mu: float = 10.0,
    t0: float = 1.0,
    eps: float = 0.01,
    max_iter: int = 100
) -> Tuple[np.ndarray, float, Dict]:

    A = problem['A']

    b = problem['b']

    q_star = problem['q_star']

    p = problem['p']

    n = problem['n']

    m = n

    nu = nu0.copy()

    lam = np.ones(n)

    z = np.concatenate([nu, lam])

    t = t0

    history = {
        'outer_iter': [],
        'inner_iter': [],
        'objective': [],
        'error': [],
        't_values': [],
        'total_newton_steps': 0
    }

    total_newton_steps = 0

    for outer_iter in range(max_iter):

        inner_iter = 0

        for _ in range(50):

            grad = dual_phi_grad(
                z,
                A,
                b,
                t
            )

            H = dual_phi_hess(
                z,
                A,
                t
            )

            H += 1e-10 * np.eye(H.shape[0])

            try:

                dz = -np.linalg.solve(H, grad)

            except np.linalg.LinAlgError:
                break

            newton_decrement = np.sqrt(
                np.abs(grad @ dz)
            )

            if newton_decrement < 1e-8:
                break

            step = line_search_scalar(
                f=lambda z_: dual_phi(z_, A, b, t),
                x=z,
                p=dz,
                gamma=1.0,
                min_gamma=1e-8,
                num=30
            )

            if step <= 1e-12:
                break

            z = z + step * dz

            z[p:] = np.maximum(
                z[p:],
                1e-12
            )

            inner_iter += 1

            total_newton_steps += 1

        nu = z[:p]

        lam = z[p:]

        exp_term = np.exp(
            -1 - A.T @ nu + lam
        )

        phi_val = (
            b @ nu
            - np.sum(lam)
            + np.sum(exp_term)
        )

        q_val = -phi_val

        error = abs(q_val - q_star)

        duality_gap = m / t

        history['outer_iter'].append(outer_iter)
        history['inner_iter'].append(inner_iter)
        history['objective'].append(q_val)
        history['error'].append(error)
        history['t_values'].append(t)
        history['total_newton_steps'] = total_newton_steps

        if error <= eps:

            print(
                f"    Dual converged at outer iter "
                f"{outer_iter}, error: {error:.2e}"
            )

            break

        t *= mu

    return z, q_val, history

# Сравнение

In [11]:
def run_experiments(
    problems,
    primal_solver=None,
    dual_solver=None,
    method_name="method",
    save_dir="results",
    verbose=True,
    mu: float = 10.0,
    t0: float = 1.0,
    eps: float = 0.01,
    max_iter: int = 100,
):

    save_dir = Path(save_dir)

    save_dir.mkdir(exist_ok=True)

    results = []

    for problem in problems:

        problem_id = problem["problem_id"]

        n = problem["n"]

        instance_id = problem["instance_id"]

        num_starts = len(problem["primal_starts"])

        for start_id in range(num_starts):

            row = {
                "method": method_name,
                "problem_id": problem_id,
                "n": n,
                "p": problem["p"],
                "instance_id": instance_id,
                "start_id": start_id,
                "mu": mu,
                "t0": t0,
                "eps": eps,
                "max_iter": max_iter,
            }


            if primal_solver is not None:

                x0 = problem["primal_starts"][start_id]

                t0_time = time.time()

                x_sol, f_val, hist_p = primal_solver(
                    problem,
                    x0,
                    mu=mu,
                    t0=t0,
                    eps=eps,
                    max_iter=max_iter
                )

                time_p = time.time() - t0_time

                final_error = (
                    hist_p['error'][-1]
                    if hist_p['error']
                    else np.inf
                )

                outer_iters = len(hist_p['outer_iter'])

                total_steps = hist_p.get(
                    'total_newton_steps',
                    hist_p.get('total_bfgs_steps', 0)
                )

                row.update({
                    "primal_solution": x_sol,
                    "primal_iters": outer_iters,
                    "primal_inner_steps": total_steps,
                    "primal_time": time_p,
                    "primal_final_error": final_error,
                    "primal_success": final_error <= eps,
                    "primal_final_objective": f_val,
                    "primal_history": hist_p,
                })


            if dual_solver is not None:

                nu0 = problem["dual_starts"][start_id]

                t0_time = time.time()

                z_sol, q_val, hist_d = dual_solver(
                    problem,
                    nu0,
                    mu=mu,
                    t0=t0,
                    eps=eps,
                    max_iter=max_iter
                )

                time_d = time.time() - t0_time

                p_dim = problem["p"]

                nu_sol = z_sol[:p_dim]

                lam_sol = z_sol[p_dim:]

                x_from_dual = recover_primal_from_dual(
                    nu_sol,
                    lam_sol,
                    problem["A"]
                )

                final_error = (
                    hist_d['error'][-1]
                    if hist_d['error']
                    else np.inf
                )

                outer_iters = len(hist_d['outer_iter'])

                total_steps = hist_d.get(
                    'total_newton_steps',
                    hist_d.get('total_bfgs_steps', 0)
                )

                row.update({
                    "dual_solution": z_sol,
                    "dual_nu": nu_sol,
                    "dual_lambda": lam_sol,
                    "dual_recovered_x": x_from_dual,
                    "dual_iters": outer_iters,
                    "dual_inner_steps": total_steps,
                    "dual_time": time_d,
                    "dual_final_error": final_error,
                    "dual_success": final_error <= eps,
                    "dual_final_objective": q_val,
                    "dual_history": hist_d,
                })

            results.append(row)

            if verbose:

                msg = (
                    f"{method_name} | "
                    f"n={n} | "
                    f"inst={instance_id} | "
                    f"start={start_id} | "
                    f"μ={mu}, t0={t0}"
                )

                if primal_solver is not None:

                    msg += (
                        f" | primal_it={row['primal_iters']}"
                        f" ({row['primal_inner_steps']} inner), "
                        f"time={row['primal_time']:.4f}s, "
                        f"err={row['primal_final_error']:.2e}"
                    )

                if dual_solver is not None:

                    msg += (
                        f" | dual_it={row['dual_iters']}"
                        f" ({row['dual_inner_steps']} inner), "
                        f"time={row['dual_time']:.4f}s, "
                        f"err={row['dual_final_error']:.2e}"
                    )

                print(msg)


    pickle_path = save_dir / f"{method_name}_results.pkl"

    with open(pickle_path, "wb") as f:
        pickle.dump(results, f)

    print(f"\nPickle results saved to {pickle_path}")

    return results

In [12]:
newton_results = run_experiments(
    problems=problems,
    primal_solver=barrier_method_newton,
    dual_solver=barrier_method_dual_newton,
    method_name="barrier_newton",
    save_dir="results",
    verbose=True,
    mu=10.0,
    t0=1.0,
    eps=0.01,
    max_iter=100,
)

    Dual converged at outer iter 3, error: 5.00e-03
barrier_newton | n=10 | inst=0 | start=0 | μ=10.0, t0=1.0 | primal_it=4 (29 inner), time=0.0701s, err=5.00e-03 | dual_it=4 (31 inner), time=0.0697s, err=5.00e-03
    Dual converged at outer iter 3, error: 5.00e-03
barrier_newton | n=10 | inst=0 | start=1 | μ=10.0, t0=1.0 | primal_it=4 (32 inner), time=0.0505s, err=5.00e-03 | dual_it=4 (29 inner), time=0.0464s, err=5.00e-03
    Dual converged at outer iter 3, error: 5.00e-03
barrier_newton | n=10 | inst=0 | start=2 | μ=10.0, t0=1.0 | primal_it=4 (30 inner), time=0.0368s, err=5.00e-03 | dual_it=4 (32 inner), time=0.0596s, err=5.00e-03
    Dual converged at outer iter 3, error: 5.00e-03
barrier_newton | n=10 | inst=0 | start=3 | μ=10.0, t0=1.0 | primal_it=4 (32 inner), time=0.0527s, err=5.00e-03 | dual_it=4 (31 inner), time=0.0581s, err=5.00e-03
    Dual converged at outer iter 3, error: 5.00e-03
barrier_newton | n=10 | inst=0 | start=4 | μ=10.0, t0=1.0 | primal_it=4 (33 inner), time=0.0